In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Forschungsseminar_Computergestützte_Analyse_Sozialer_Medien/Daten/RAW/Data_export_with_country_names.csv', delimiter=',')


In [3]:
df.head()

,Unnamed: 0,id,thread_id,parent_id,body,author,author_fullname,author_avatar_url,timestamp,type,...,media_url,hashtags,num_likes,num_comments,num_media,location_name,location_latlong,location_city,unix_timestamp,country
0,0,Copte3RpP6p,Copte3RpP6p,Copte3RpP6p,Happy Valentine’s Day honey 🌹\nI absolutely ad...,esteecwilliams,Estee Williams,https://scontent-ham3-1.cdninstagram.com/v/t51...,2/14/23 17:47,photo,...,https://scontent-ham3-1.cdninstagram.com/v/t51...,NaN,913,0,4,Virginia,"37.5,-79",NaN,1676396852,usa
1,1,CodAJHDJJva,CodAJHDJJva,CodAJHDJJva,So thankful for my husband who has given me th...,esteecwilliams,Estee Williams,https://scontent-ham3-1.cdninstagram.com/v/t51...,2/9/23 19:20,photo,...,https://scontent-ham3-1.cdninstagram.com/v/t51...,"tradwife,traditionalmarriage,homemaker,wifey,b...",942,0,9,Virginia,"37.5,-79",NaN,1675970427,usa
2,2,CoFRe6mOic6,CoFRe6mOic6,CoFRe6mOic6,Chana Masala \n\nThis recipe is one I learned ...,esteecwilliams,Estee Williams,https://scontent-ham3-1.cdninstagram.com/v/t51...,1/31/23 14:10,mixed,...,https://scontent-ham3-1.cdninstagram.com/v/t51...,"cookwithesteewilliams,indianrecipes,chanamasal...",424,0,2,Virginia,"37.5,-79",NaN,1675174212,usa
3,3,CnxR-6YJYGm,CnxR-6YJYGm,CnxR-6YJYGm,The moment when mama’s dress fit like a glove 🥰,esteecwilliams,Estee Williams,https://scontent-ham3-1.cdninstagram.com/v/t51...,1/23/23 19:49,photo,...,https://scontent-ham3-1.cdninstagram.com/v/t51...,NaN,728,0,2,Virginia,"37.5,-79",NaN,1674503386,usa
4,4,CnpCJQILJtR,CnpCJQILJtR,CnpCJQILJtR,Spicy venison lasagna \nStart by boiling lasag...,esteecwilliams,Estee Williams,https://scontent-ham3-1.cdninstagram.com/v/t51...,1/20/23 14:57,mixed,...,https://scontent-ham3-1.cdninstagram.com/v/t51...,"venisonrecipes,esteeinthekitchen",358,0,4,Virginia,"37.5,-79",NaN,1674226647,usa


**$\chi^2$-Test: Ist die *Setting* variable in beiden Gruppen (*type*=Story vs Post) gleich verteilt?**

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

print("DataFrame columns:", df.columns)

# contingency table
ct = pd.crosstab(df['type'], df['country'])

# chi-square test
chi2, p, dof, expected = chi2_contingency(ct)

# Cramer's V
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

print(f"Chi-square: {chi2:.3f}")
print(f"df: {dof}")
print(f"p-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

DataFrame columns: Index(['Unnamed: 0', 'id', 'thread_id', 'parent_id', 'body', 'author',
       'author_fullname', 'author_avatar_url', 'timestamp', 'type', 'url',
       'image_url', 'media_url', 'hashtags', 'num_likes', 'num_comments',
       'num_media', 'location_name', 'location_latlong', 'location_city',
       'unix_timestamp', 'country'],
      dtype='object')
Chi-square: 16.613
df: 2
p-value: 0.0002
Cramer's V: 0.190


In [5]:
desc = (
    df
    .groupby(['type', 'country'])
    .size()
    .reset_index(name='n')
)

desc['percent'] = (
    desc
    .groupby('type')['n']
    .transform(lambda x: x / x.sum() * 100)
)

desc

,type,country,n,percent
0,mixed,usa,17,100.00000
1,photo,germany,105,43.38843
2,photo,usa,137,56.61157
3,video,germany,101,50.50000
4,video,usa,99,49.50000


Funktioniert auch mit z. B. binären Variablen:

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact

# Ensure collage is clean binary (0/1). Adjust mapping if needed.
# If it's already 0/1 or boolean, this will usually work as-is.
collage = df['country']
# The following mapping is incorrect for 'country' values like 'usa', 'germany'.
# If a binary variable is needed, the mapping keys should match actual values,
# or a different binary variable should be selected.
# if collage.dtype == 'O':  # object, likely strings
#    df = df.copy()
#    df['country'] = df['country'].str.strip().str.lower().map({'yes': 1, 'no': 0, 'true': 1, 'false': 0})

# 1) Chi-square test of independence
# Tests whether collage proportions differ across 'type' categories.
ct = pd.crosstab(df['type'], df['country'])

chi2, p, dof, expected = chi2_contingency(ct)

In [7]:
# Cramér’s V is a standardized association measure for contingency tables.
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

# If expected counts are small (rule of thumb: any < 5), Fisher’s exact is preferred for 2x2 tables.
min_expected = expected.min()

print("Contingency table (type x collage):")
print(ct)
print()
print(f"Chi-square: {chi2:.3f} | df: {dof} | p-value: {p:.4f} | Cramer's V: {cramers_v:.3f}")
print(f"Min expected cell count: {min_expected:.2f}")

# Fisher's exact test for 2x2 only
if ct.shape == (2, 2) and min_expected < 5:
    oddsratio, p_fisher = fisher_exact(ct.values)
    print(f"Fisher's exact p-value: {p_fisher:.4f} | odds ratio: {oddsratio:.3f}")

Contingency table (type x collage):
country  germany  usa
type                 
mixed          0   17
photo        105  137
video        101   99

Chi-square: 16.613 | df: 2 | p-value: 0.0002 | Cramer's V: 0.190
Min expected cell count: 7.63


**Logistische Regression: Wie beeinflusst die Gruppierungsvariable *type* die Wahrscheinlichkeit, dass ein Bild eine *Collage* ist**

In [8]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Logistic regression models the probability of being 'usa' (1) vs other countries (0) as a function of 'type'.

# Make sure the dependent variable is numeric 0/1 for statsmodels
df_lr = df.dropna(subset=['country', 'type']).copy()

# Convert 'country' to a binary variable: 1 if 'usa', 0 otherwise
df_lr['is_usa'] = (df_lr['country'] == 'usa').astype(int)

# Baseline model: is_usa ~ type (type treated as categorical automatically via C())
model = smf.logit("is_usa ~ C(type)", data=df_lr).fit(disp=False)
print(model.summary())

# Odds ratios are easier to interpret than log-odds coefficients.
params = model.params
conf = model.conf_int()
or_table = pd.DataFrame({
    "OR": np.exp(params),
    "CI_low": np.exp(conf[0]),
    "CI_high": np.exp(conf[1]),
    "p": model.pvalues
})
print("\nOdds ratios (reference category is the first level of type):")
print(or_table)

                           Logit Regression Results                           
Dep. Variable:                 is_usa   No. Observations:                  459
Model:                          Logit   Df Residuals:                      456
Method:                           MLE   Df Model:                            2
Date:                Fri, 27 Feb 2026   Pseudo R-squ.:                    -inf
Time:                        09:59:19   Log-Likelihood:                   -inf
converged:                      False   LL-Null:                       -315.74
Covariance Type:            nonrobust   LLR p-value:                     1.000
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept        -7.407e+08        nan        nan        nan         nan         nan
C(type)[T.photo]  7.407e+08        nan        nan        nan         nan         nan
C(type)[T.video]  7.407e+08 

/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


Logistic regression indicated that Stories were less likely to be collages than Posts (OR = 0.77, 95% CI [0.67, 0.89], p < .001), although the effect size was small (pseudo R² = .003).

**Cross-National Setting Analyse**

In [9]:
df = pd.read_csv('/content/drive/MyDrive/Forschungsseminar_Computergestützte_Analyse_Sozialer_Medien/Daten/classification_results.csv')

In [10]:
# Unsure ausgeschlossen:
df_setting = df[df['Majority Decision'] != 'Unsure'].copy()

# Kreuztabelle
ct_setting = pd.crosstab(df_setting['Majority Decision'], df_setting['country'])
print("Contingency table (Setting x Country):")
print(ct_setting)

# Chi-Square
chi2, p, dof, expected = chi2_contingency(ct_setting)

# Cramér’s V
n = ct_setting.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct_setting.shape) - 1)))

# Prozentwerte innerhalb der Länder (Spaltenprozente)
percent_setting = ct_setting.div(ct_setting.sum(axis=0), axis=1) * 100

print("\nRel. percent distribution:")
print(percent_setting.round(1))
print("\nChi-square:", round(chi2, 3))
print("df:", dof)
print("p-value:", round(p, 6))
print("Cramér’s V:", round(cramers_v, 3))

Contingency table (Setting x Country):
country            germany  usa
Majority Decision              
Inside                  90  121
NotApplicable           34   71
Outside                 75   47

Rel. percent distribution:
country            germany   usa
Majority Decision               
Inside                45.2  50.6
NotApplicable         17.1  29.7
Outside               37.7  19.7

Chi-square: 20.537
df: 2
p-value: 3.5e-05
Cramér’s V: 0.217
